# Preparação do corpus para BERTilda — ParlaMint-PT

Este notebook prepara o corpus ao nível da intervenção para o BERTilda. Cada linha corresponde a uma intervenção e cada janela temporal corresponde a um ano.

## Decisões de desenho

- Unidade documental: intervenção parlamentar.
- Índice temporal: ano.
- Janela inicial: 1 ano, sem sobreposição.
- Metadados políticos são preservados para interpretação posterior; não entram no texto do modelo.

In [4]:
from pathlib import Path
import re

import pandas as pd

## 1. Carregamento do corpus pré-processado

In [5]:
possible_paths = [
    Path("../../data/parlamint_pt_merged.csv"),
    Path("data/parlamint_pt_merged.csv"),
]

input_file = next((path for path in possible_paths if path.exists()), None)

if input_file is None:
    raise FileNotFoundError("Não foi encontrada a tabela pré-processada.")

metadata = pd.read_csv(input_file, dtype="string")

required_columns = {"ID", "Date", "texto", "Speaker_ID", "bloco_politico"}
missing_columns = required_columns - set(metadata.columns)

if missing_columns:
    raise ValueError(f"Faltam colunas necessárias: {sorted(missing_columns)}")

metadata["Date"] = pd.to_datetime(metadata["Date"], errors="coerce")
metadata["ano"] = metadata["Date"].dt.year.astype("Int64")

print(f"Ficheiro: {input_file}")
print(f"Intervenções carregadas: {len(metadata):,}")

Ficheiro: ../../data/parlamint_pt_merged.csv
Intervenções carregadas: 248,573


## 2. Preparação do texto

O texto original é preservado. `Tokens` é a representação simples usada pelo pipeline inicial do BERTilda; nesta fase não removemos stopwords nem aplicamos lematização.

In [6]:
def tokenizar_texto(texto):
    texto = "" if pd.isna(texto) else str(texto)
    texto = re.sub(r"\[\[.*?\]\]", " ", texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " ", texto)
    texto = texto.lower()
    return re.findall(r"\b[\wÀ-ÖØ-öø-ÿ]+(?:-[\wÀ-ÖØ-öø-ÿ]+)*\b", texto)

metadata["Tokens"] = metadata["texto"].apply(tokenizar_texto)
metadata["n_tokens"] = metadata["Tokens"].str.len()

corpus_bertilda = metadata.loc[
    metadata["ano"].notna() & metadata["n_tokens"].ge(5)
].copy()

corpus_bertilda["documento_modelo"] = corpus_bertilda["Tokens"].str.join(" ")

print(f"Intervenções excluídas por terem menos de 5 tokens: {len(metadata) - len(corpus_bertilda):,}")
print(f"Documentos disponíveis para BERTilda: {len(corpus_bertilda):,}")

Intervenções excluídas por terem menos de 5 tokens: 44,376
Documentos disponíveis para BERTilda: 204,197


## 3. Tabela de entrada anual

`corpus_bertilda` mantém os metadados necessários para interpretar os tópicos depois da modelação. O BERTilda usará `ano` e `documento_modelo`; `ID` permite voltar à intervenção original.

In [ ]:
colunas_modelo = [
    "ID", "Text_ID", "Date", "ano", "texto", "documento_modelo", "Tokens",
    "Speaker_ID", "Speaker_name", "Speaker_party", "bloco_politico",
    "Speaker_minister", "Topic", "n_tokens",
]

corpus_bertilda = corpus_bertilda[colunas_modelo].copy()

cobertura_anual = (
    corpus_bertilda
    .groupby("ano")
    .agg(
        documentos=("ID", "size"),
        tokens=("n_tokens", "sum"),
        oradores=("Speaker_ID", "nunique"),
    )
    .sort_index()
)

cobertura_anual

,documentos,tokens,oradores
ano,,,
2015,18442,2262407,335
2016,20418,2410803,270
2017,21901,2469114,277
2018,22752,2555835,289
2019,21850,2331597,352
2020,21500,2803635,289
2021,16490,2589033,283
2022,19781,2224995,294
2023,38698,3428947,284


## 4. Guardar a entrada do BERTilda

O ficheiro guardado será usado no notebook de modelação.

In [8]:
project_dir = input_file.parent.parent
output_dir = project_dir / "data"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "parlamint_pt_bertilda_input.csv"
corpus_bertilda.to_csv(output_file, index=False, encoding="utf-8")

print(f"Corpus BERTilda guardado em: {output_file}")
print(f"Documentos: {len(corpus_bertilda):,}")

Corpus BERTilda guardado em: ../../data/parlamint_pt_bertilda_input.csv
Documentos: 204,197


## Próximo passo

No notebook de modelação, usar `documento_modelo` como documento, `ano` para formar janelas de 1 ano e o mesmo modelo de embeddings em todos os anos.